In [1]:
# Cell 1: Mount Drive and set workspace
import os
from google.colab import drive

drive.mount("/content/drive", force_remount=True)

def find_baseline_dir():
    candidates = [
        "/content/drive/MyDrive/final_project/baseline",
        "/content/drive/MyDrive/final_project/baseline/",
    ]
    for p in candidates:
        if os.path.isdir(p):
            return os.path.abspath(p)

    shared_root = "/content/drive/Shareddrives"
    if os.path.isdir(shared_root):
        for root, dirs, _ in os.walk(shared_root):
            if root.endswith("/final_project") and "baseline" in dirs:
                return os.path.abspath(os.path.join(root, "baseline"))

    raise FileNotFoundError("Could not find final_project/baseline in Drive.")

BASE_DIR = find_baseline_dir()
os.chdir(BASE_DIR)

print("BASE_DIR =", BASE_DIR)
print("CWD =", os.getcwd())

Mounted at /content/drive
BASE_DIR = /content/drive/MyDrive/final_project/baseline
CWD = /content/drive/.shortcut-targets-by-id/1V7smEWLD_ZhlaD773UjRiHThZZ9cpgS-/final_project/baseline


In [2]:
# Cell 2: Install gpt-oss dependencies
import sys
import subprocess
import os

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-U", "pip", "setuptools", "wheel", "-q"
])

# Remove packages that often create CUDA import conflicts in Colab.
subprocess.call(
    [sys.executable, "-m", "pip", "uninstall", "-y", "bitsandbytes", "torchvision", "torchaudio"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

# gpt-oss model card recommends transformers + kernels + torch.
pkgs = [
    "torch",
    "transformers",
    "kernels",
    "accelerate",
    "huggingface-hub",
    "safetensors",
    "tqdm==4.66.2",
]

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U"] + pkgs)

# Clear stale imports if the runtime previously loaded conflicting packages.
for m in list(sys.modules.keys()):
    if m.startswith("torchvision") or m.startswith("bitsandbytes"):
        del sys.modules[m]

import torch
import transformers
import accelerate
import huggingface_hub
import tqdm

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)
print("huggingface_hub:", huggingface_hub.__version__)
print("tqdm:", tqdm.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

print("Installed OK")

torch: 2.11.0+cu130
transformers: 5.8.0
accelerate: 1.13.0
huggingface_hub: 1.14.0
tqdm: 4.66.2
CUDA available: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
GPU memory GB: 94.97
Installed OK


In [3]:
# Cell 3: Clone or update repository
import os
import subprocess

REPO_URL = "https://github.com/ali-mohmmadi/KGP-CuriousLLM.git"
REPO_DIR = "/content/KGP-CuriousLLM"

if not os.path.isdir(REPO_DIR):
    print("Cloning repository into:", REPO_DIR)
    subprocess.check_call(["git", "clone", REPO_URL, REPO_DIR])
else:
    print("Repo already exists. Pulling latest changes...")
    subprocess.check_call(["git", "-C", REPO_DIR, "pull"])

print("Repo ready at:", REPO_DIR)
print("Repo root files:", os.listdir(REPO_DIR)[:15])

Cloning repository into: /content/KGP-CuriousLLM
Repo ready at: /content/KGP-CuriousLLM
Repo root files: ['configs', 'quantize_mistral_main.py', 'ft_mistral_main.py', 'KGP', 'MDR_main.py', 'MDR_embedding_main.py', '.git', 'README.md', 'T5_main.py', 'grid_search_mistral_main.py', 'images', 'requirements.txt', 'create_dirs.py', 'kgp_main.py', '.gitignore']


In [4]:
# Cell 4: Add repo and baseline to Python path
import os
import sys

if BASE_DIR not in sys.path:
    sys.path.insert(0, BASE_DIR)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("PYTHONPATH ready")
print("BASE_DIR in path:", BASE_DIR in sys.path)
print("REPO_DIR in path:", REPO_DIR in sys.path)

PYTHONPATH ready
BASE_DIR in path: True
REPO_DIR in path: True


In [5]:
# Cell 5: Set HotpotQA paths and gpt-oss config
import os

MODEL_NAME = "openai/gpt-oss-120b"
REASONING_LEVEL = "low"

DATA_PATH = os.path.join(
    BASE_DIR,
    "DATA",
    "KG",
    "evidence",
    "hotpot_evidence_1000",
    "qwen_agent",
    "evidence.json",
)

SAVE_DIR = os.path.join(
    BASE_DIR,
    "DATA",
    "KG",
    "answers",
    "hotpot_answers_gpt_oss_120b",
)

SAVE_PATH = os.path.join(
    SAVE_DIR,
    "qwen_agent_responses.json",
)

os.makedirs(SAVE_DIR, exist_ok=True)

print("MODEL_NAME =", MODEL_NAME)
print("REASONING_LEVEL =", REASONING_LEVEL)
print("DATA_PATH  =", DATA_PATH)
print("SAVE_DIR   =", SAVE_DIR)
print("SAVE_PATH  =", SAVE_PATH)

assert os.path.isfile(DATA_PATH), f"Missing evidence file: {DATA_PATH}"
print("Evidence file exists.")

MODEL_NAME = openai/gpt-oss-120b
REASONING_LEVEL = low
DATA_PATH  = /content/drive/MyDrive/final_project/baseline/DATA/KG/evidence/hotpot_evidence_1000/qwen_agent/evidence.json
SAVE_DIR   = /content/drive/MyDrive/final_project/baseline/DATA/KG/answers/hotpot_answers_gpt_oss_120b
SAVE_PATH  = /content/drive/MyDrive/final_project/baseline/DATA/KG/answers/hotpot_answers_gpt_oss_120b/qwen_agent_responses.json
Evidence file exists.


In [6]:
# Cell 6: Check GPU memory
import torch

assert torch.cuda.is_available(), "GPU is required for gpt-oss-120b."

gpu_name = torch.cuda.get_device_name(0)
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3

print("GPU:", gpu_name)
print("GPU memory GB:", round(gpu_mem_gb, 2))

assert gpu_mem_gb >= 75, (
    f"gpt-oss-120b is expected to need about an 80GB GPU. "
    f"Current GPU memory is only {gpu_mem_gb:.2f} GB."
)

print("GPU memory looks sufficient.")

GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
GPU memory GB: 94.97
GPU memory looks sufficient.


In [7]:
# Cell 7: Load HotpotQA evidence
import json
from collections import Counter

def load_json(file_path: str):
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data

data = load_json(DATA_PATH)

print("num_records =", len(data))
print("type_counts =", Counter(r.get("type", "unknown") for r in data))
print("first_keys =", list(data[0].keys()))

print("\nFirst question:")
print(data[0]["question"])

print("\nFirst answer:")
print(data[0].get("answer", ""))

print("\nFirst evidence preview:")
for i, ev in enumerate(data[0].get("evidence", [])[:3], start=1):
    print(f"{i}.", ev[:500])

num_records = 1000
type_counts = Counter({'bridge': 700, 'comparison': 300})
first_keys = ['type', 'question', 'evidence', 'answer', 'supports']

First question:
Where operation Operation Dragoon and Battle of Cold Harbor fought during to different wars?

First answer:
yes

First evidence preview:
1. Title: Operation Dragoon. Evidence: Operation Dragoon also had political implications.
2. Title: Operation Dragoon. Evidence: Despite these successes, there was criticism of Dragoon by some Allied generals and contemporary commentators such as Bernard Montgomery, Arthur R. Wilson, and Chester Wilmot in the aftermath, mostly because of its geo-strategic implications.
3. Title: Operation Dragoon. Evidence: In the northeast the German problems loomed as large.


In [8]:
# Cell 8: Define repo-faithful GPT prompt

prompt = """
    Given the question and its associated contexts below, please generate a concise, precise answer in English. The answer must strictly adhere to the following guidelines:

    - The answer should be directly relevant to the question.
    - Provide the answer in a clear, straightforward format.
    - Limit your answer to no more than 6 words, focusing on the essential information requested.
    - If the provided contexts do not contain enough information to answer the question, respond with "Information not available" or "Cannot determine from provided context."
    - Do not include any additional tokens, explanations, or information beyond the direct answer.
    - Carefully reason through the question and contexts if the question involes time.

    QUESTION: {question}
    CONTEXT: {context}
    ANSWER: [Your concise answer here or "Information not available" if the answer cannot be determined from the contexts.]

    """

none_prompt = """Given the following question and contexts, create a final answer in English to the question.
    QUESTION: {question}
    ANSWER: [Please provide only the answer and keep the answer less than 6 words.]
    """

print("Prompts ready.")

Prompts ready.


In [9]:
# Cell 9: Load gpt-oss-120b with direct generate interface
import os
import gc
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers.utils import logging as hf_logging
import warnings

MODEL_NAME = "openai/gpt-oss-120b"

# Reduce noisy generation warnings.
hf_logging.set_verbosity_error()
warnings.filterwarnings("ignore", message=".*Both `max_new_tokens`.*")
warnings.filterwarnings("ignore", message=".*Passing `generation_config`.*")
warnings.filterwarnings("ignore", message=".*clean_up_tokenization_spaces.*")

# If the previous pipeline already loaded the model, reuse it.
if "pipe" in globals():
    print("Reusing model and tokenizer from existing pipeline.")
    tokenizer = pipe.tokenizer
    model = pipe.model
    del pipe
    gc.collect()
    torch.cuda.empty_cache()
else:
    print("Loading model and tokenizer from Hugging Face.")
    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME,
        trust_remote_code=True,
    )

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        dtype="auto",
        device_map="auto",
        trust_remote_code=True,
    )

model.eval()

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loaded model:", MODEL_NAME)
print("Tokenizer loaded:", tokenizer.__class__.__name__)
print("Model dtype sample:", next(model.parameters()).dtype)
print("Model device sample:", next(model.parameters()).device)

if hasattr(model, "hf_device_map"):
    print("Device map available.")

Loading model and tokenizer from Hugging Face.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/27.9M [00:00<?, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/kernels/utils.py:401: FutureWarning: Future versions of `kernels` (>=0.15) will require specifying a kernel version or revision. See: https://huggingface.co/docs/kernels/migration
  revision = select_revision_or_version(repo_id, revision=revision, version=version)


Fetching ... files: 0it [00:00, ?it/s]

Loading weights:   0%|          | 0/615 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/177 [00:00<?, ?B/s]

Loaded model: openai/gpt-oss-120b
Tokenizer loaded: TokenizersBackend
Model dtype sample: torch.bfloat16
Model device sample: cuda:0


In [10]:
# Cell 10: Define gpt-oss final-channel parser and cleaner
import re

def clean_gpt_oss_response(text: str) -> str:
    if text is None:
        return ""

    text = str(text).strip()

    # Remove possible tags or special leftovers.
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
    text = re.sub(r"<analysis>.*?</analysis>", "", text, flags=re.DOTALL).strip()

    # Remove harmony-like special tokens if still visible.
    special_tokens = [
        "<|start|>", "<|end|>", "<|return|>",
        "<|message|>", "<|channel|>",
        "<|assistant|>", "<|user|>", "<|system|>",
    ]
    for tok in special_tokens:
        text = text.replace(tok, " ")

    text = re.sub(r"\s+", " ", text).strip()

    # Remove common answer prefixes.
    for prefix in ["ANSWER:", "Answer:", "answer:"]:
        if text.startswith(prefix):
            text = text[len(prefix):].strip()

    # Keep first sentence-like line only.
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    if lines:
        text = lines[0].strip()

    text = text.strip().strip('"').strip("'").strip()

    return text


def extract_final_from_raw_completion(raw_text: str):
    if raw_text is None:
        return None

    raw = str(raw_text).strip()

    # Best case: raw decode still contains harmony channel markers.
    patterns = [
        r"<\|channel\|>final<\|message\|>(.*?)(?:<\|end\|>|<\|return\|>|$)",
        r"final<\|message\|>(.*?)(?:<\|end\|>|<\|return\|>|$)",
        r"<\|final\|>(.*?)(?:<\|end\|>|<\|return\|>|$)",
    ]

    for pat in patterns:
        m = re.search(pat, raw, flags=re.DOTALL)
        if m:
            return clean_gpt_oss_response(m.group(1))

    # Fallback for decoded text where special tokens disappeared.
    # Example: "... analysis ... final yes"
    lowered = raw.lower()
    final_pos = lowered.rfind("final")
    analysis_pos = lowered.find("analysis")

    if final_pos != -1 and final_pos > analysis_pos:
        candidate = raw[final_pos + len("final"):]
        return clean_gpt_oss_response(candidate)

    # If only analysis is present, do not return it as answer.
    if lowered.startswith("analysis") or "we need to answer" in lowered[:300]:
        return None

    return clean_gpt_oss_response(raw)


def is_bad_gpt_oss_response(text: str) -> bool:
    if text is None:
        return True

    s = str(text).strip()
    low = s.lower()

    if not s:
        return True

    if low.startswith("analysis"):
        return True

    if "we need to answer" in low[:300]:
        return True

    if "<|channel|>analysis" in low:
        return True

    return False

print("Parser and cleaner ready.")

Parser and cleaner ready.


In [11]:
# Cell 11: Define gpt-oss low-reasoning generation with direct generate
import torch

def build_gpt_oss_inputs(input_prompt: str):
    messages = [
        {
            "role": "system",
            "content": "You are a QA generation assistant.",
        },
        {
            "role": "user",
            "content": input_prompt,
        },
    ]

    input_device = next(model.parameters()).device

    # Correct gpt-oss reasoning control.
    try:
        inputs = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            return_tensors="pt",
            return_dict=True,
            reasoning_effort=REASONING_LEVEL,
        )
    except TypeError:
        # Fallback if this tokenizer version does not support reasoning_effort.
        messages[0]["content"] = f"Reasoning: {REASONING_LEVEL}\nYou are a QA generation assistant."
        inputs = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            return_tensors="pt",
            return_dict=True,
        )

    inputs = inputs.to(input_device)
    return inputs


@torch.inference_mode()
def gpt_oss_chat_completion(input_prompt: str, max_new_tokens: int = 256):
    inputs = build_gpt_oss_inputs(input_prompt)

    generated = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

    completion_ids = generated[0][inputs["input_ids"].shape[-1]:]

    # Keep special tokens so we can detect harmony final channel.
    raw_completion = tokenizer.decode(
        completion_ids,
        skip_special_tokens=False,
        clean_up_tokenization_spaces=False,
    )

    final_answer = extract_final_from_raw_completion(raw_completion)

    # If the model did not reach final channel, retry with more tokens once.
    if final_answer is None or is_bad_gpt_oss_response(final_answer):
        generated = model.generate(
            **inputs,
            max_new_tokens=max(max_new_tokens * 2, 512),
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

        completion_ids = generated[0][inputs["input_ids"].shape[-1]:]
        raw_completion = tokenizer.decode(
            completion_ids,
            skip_special_tokens=False,
            clean_up_tokenization_spaces=False,
        )

        final_answer = extract_final_from_raw_completion(raw_completion)

    if final_answer is None or is_bad_gpt_oss_response(final_answer):
        return "Information not available"

    return final_answer


print("gpt-oss low-reasoning generation function ready.")

gpt-oss low-reasoning generation function ready.


In [12]:
# Cell 11A: Debug raw gpt-oss completion if needed
import torch

@torch.inference_mode()
def debug_gpt_oss_raw_completion(input_prompt: str, max_new_tokens: int = 256):
    inputs = build_gpt_oss_inputs(input_prompt)

    generated = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

    completion_ids = generated[0][inputs["input_ids"].shape[-1]:]
    raw_completion = tokenizer.decode(
        completion_ids,
        skip_special_tokens=False,
        clean_up_tokenization_spaces=False,
    )

    parsed = extract_final_from_raw_completion(raw_completion)

    print("RAW COMPLETION:")
    print(raw_completion[:3000])
    print("\nPARSED FINAL:")
    print(parsed)

    return raw_completion, parsed

print("Debug function ready.")

Debug function ready.


In [13]:
# Cell 12: Build one HotpotQA input prompt

sample = data[0]

sample_type = sample["type"]
sample_question = sample["question"]
sample_contexts = sample["evidence"]
sample_gt = sample["answer"]

if sample_contexts:
    sample_contexts_text = "\n".join(
        f"{i}: {c}" for i, c in enumerate(sample_contexts, start=1)
    )
    sample_input_prompt = prompt.format(
        question=sample_question,
        context=sample_contexts_text,
    )
else:
    sample_input_prompt = none_prompt.format(question=sample_question)

print("Sample type:", sample_type)
print("Sample question:", sample_question)
print("Sample gt:", sample_gt)
print("\nPrompt preview:")
print(sample_input_prompt[:3000])

Sample type: comparison
Sample question: Where operation Operation Dragoon and Battle of Cold Harbor fought during to different wars?
Sample gt: yes

Prompt preview:

    Given the question and its associated contexts below, please generate a concise, precise answer in English. The answer must strictly adhere to the following guidelines:

    - The answer should be directly relevant to the question.
    - Provide the answer in a clear, straightforward format.
    - Limit your answer to no more than 6 words, focusing on the essential information requested.
    - If the provided contexts do not contain enough information to answer the question, respond with "Information not available" or "Cannot determine from provided context."
    - Do not include any additional tokens, explanations, or information beyond the direct answer.
    - Carefully reason through the question and contexts if the question involes time. 

    QUESTION: Where operation Operation Dragoon and Battle of Cold Harbor f

In [14]:
# Cell 13: Smoke test gpt-oss on one HotpotQA sample

sample_response = gpt_oss_chat_completion(
    input_prompt=sample_input_prompt,
    max_new_tokens=256,
)

print("Question:", sample_question)
print("GT:", sample_gt)
print("gpt-oss response:", sample_response)

if is_bad_gpt_oss_response(sample_response):
    print("\nWARNING: Bad response detected. Run Cell 11A for raw debug.")
else:
    print("\nSmoke test output format looks OK.")

Question: Where operation Operation Dragoon and Battle of Cold Harbor fought during to different wars?
GT: yes
gpt-oss response: Information not available

Smoke test output format looks OK.


In [15]:
# Cell 14: Define repo-style answer generation pipeline
import os
import json
from tqdm import tqdm

def pipeline(data, save_path):
    responses = []

    for record in tqdm(data, total=len(data)):
        q_type = record["type"]
        question = record["question"]
        contexts = record["evidence"]
        gt = record["answer"]

        if contexts:
            contexts = "\n".join(
                f"{i}: {c}" for i, c in enumerate(contexts, start=1)
            )
            input_prompt = prompt.format(
                question=question,
                context=contexts,
            )
        else:
            input_prompt = none_prompt.format(question=question)

        resp = gpt_oss_chat_completion(
            input_prompt=input_prompt,
            max_new_tokens=256,
        )

        response = {
            "type": q_type,
            "question": question,
            "gt": gt,
            "response": resp,
        }

        responses.append(response)

        with open(save_path, "w", encoding="utf-8") as f:
            json.dump(responses, f, indent=4, ensure_ascii=False)

    return responses

print("Pipeline ready.")

Pipeline ready.


In [16]:
# Cell 15: Define repair-safe resume pipeline
import os
import json
from tqdm import tqdm

def pipeline_resume_repair(data, save_path):
    existing_by_question = {}

    if os.path.isfile(save_path):
        with open(save_path, "r", encoding="utf-8") as f:
            existing = json.load(f)

        for r in existing:
            q = r.get("question", "")
            resp = r.get("response", "")
            if q and not is_bad_gpt_oss_response(resp):
                existing_by_question[q] = r

        print("Existing responses:", len(existing))
        print("Valid existing responses kept:", len(existing_by_question))
        print("Bad existing responses to regenerate:", len(existing) - len(existing_by_question))
    else:
        print("No existing output found. Starting from scratch.")

    responses = []

    for record in tqdm(data, total=len(data)):
        q_type = record["type"]
        question = record["question"]
        contexts = record["evidence"]
        gt = record["answer"]

        if question in existing_by_question:
            responses.append(existing_by_question[question])
            continue

        if contexts:
            contexts = "\n".join(
                f"{i}: {c}" for i, c in enumerate(contexts, start=1)
            )
            input_prompt = prompt.format(
                question=question,
                context=contexts,
            )
        else:
            input_prompt = none_prompt.format(question=question)

        resp = gpt_oss_chat_completion(
            input_prompt=input_prompt,
            max_new_tokens=256,
        )

        response = {
            "type": q_type,
            "question": question,
            "gt": gt,
            "response": resp,
        }

        responses.append(response)

        with open(save_path, "w", encoding="utf-8") as f:
            json.dump(responses, f, indent=4, ensure_ascii=False)

    return responses

print("Repair-safe resume pipeline ready.")

Repair-safe resume pipeline ready.


In [17]:
# Cell 16: Run answer generation for HotpotQA

responses = pipeline_resume_repair(
    data=data,
    save_path=SAVE_PATH,
)

print("Finished.")
print("Saved to:", SAVE_PATH)
print("Total responses:", len(responses))

Existing responses: 5
Valid existing responses kept: 0
Bad existing responses to regenerate: 5


100%|██████████| 1000/1000 [49:24<00:00,  2.96s/it]

Finished.
Saved to: /content/drive/MyDrive/final_project/baseline/DATA/KG/answers/hotpot_answers_gpt_oss_120b/qwen_agent_responses.json
Total responses: 1000


In [18]:
# Cell 17: Verify saved HotpotQA answers
import os
import json
from collections import Counter

assert os.path.isfile(SAVE_PATH), f"Missing output file: {SAVE_PATH}"

saved = load_json(SAVE_PATH)

print("SAVE_PATH =", SAVE_PATH)
print("num_saved =", len(saved))
print("type_counts =", Counter(r.get("type", "unknown") for r in saved))

print("\nFirst saved response:")
print(json.dumps(saved[0], indent=2, ensure_ascii=False)[:3000])

SAVE_PATH = /content/drive/MyDrive/final_project/baseline/DATA/KG/answers/hotpot_answers_gpt_oss_120b/qwen_agent_responses.json
num_saved = 1000
type_counts = Counter({'bridge': 700, 'comparison': 300})

First saved response:
{
  "type": "comparison",
  "question": "Where operation Operation Dragoon and Battle of Cold Harbor fought during to different wars?",
  "gt": "yes",
  "response": "Information not available"
}


In [19]:
# Cell 18: Check output schema consistency

required_keys = {"type", "question", "gt", "response"}

bad_records = []
for i, record in enumerate(saved):
    if set(record.keys()) != required_keys:
        bad_records.append((i, list(record.keys())))

print("Expected keys:", required_keys)
print("Bad records:", len(bad_records))

if bad_records:
    print("First bad record:", bad_records[0])
else:
    print("All records have the expected repo-style schema.")

Expected keys: {'type', 'response', 'gt', 'question'}
Bad records: 0
All records have the expected repo-style schema.


In [20]:
# Cell 21: Final path summary

print("HotpotQA gpt-oss-120b answers:")
print(SAVE_PATH)

print("\nFile exists:")
print(os.path.isfile(SAVE_PATH))

HotpotQA gpt-oss-120b answers:
/content/drive/MyDrive/final_project/baseline/DATA/KG/answers/hotpot_answers_gpt_oss_120b/qwen_agent_responses.json

File exists:
True


In [21]:
# Cell 22: Set 2WikiMQA paths for gpt-oss-120b

import os

TWO_WIKI_DATA_PATH = os.path.join(
    BASE_DIR,
    "DATA",
    "KG",
    "evidence",
    "2wikimultihopqa_evidence_1000",
    "qwen_agent",
    "evidence.json",
)

TWO_WIKI_SAVE_DIR = os.path.join(
    BASE_DIR,
    "DATA",
    "KG",
    "answers",
    "wiki_answers_gpt_oss_120b",
)

TWO_WIKI_SAVE_PATH = os.path.join(
    TWO_WIKI_SAVE_DIR,
    "qwen_agent_responses.json",
)

os.makedirs(TWO_WIKI_SAVE_DIR, exist_ok=True)

print("TWO_WIKI_DATA_PATH =", TWO_WIKI_DATA_PATH)
print("TWO_WIKI_SAVE_DIR  =", TWO_WIKI_SAVE_DIR)
print("TWO_WIKI_SAVE_PATH =", TWO_WIKI_SAVE_PATH)

assert os.path.isfile(TWO_WIKI_DATA_PATH), f"Missing 2Wiki evidence file: {TWO_WIKI_DATA_PATH}"
print("2Wiki evidence file exists.")

TWO_WIKI_DATA_PATH = /content/drive/MyDrive/final_project/baseline/DATA/KG/evidence/2wikimultihopqa_evidence_1000/qwen_agent/evidence.json
TWO_WIKI_SAVE_DIR  = /content/drive/MyDrive/final_project/baseline/DATA/KG/answers/wiki_answers_gpt_oss_120b
TWO_WIKI_SAVE_PATH = /content/drive/MyDrive/final_project/baseline/DATA/KG/answers/wiki_answers_gpt_oss_120b/qwen_agent_responses.json
2Wiki evidence file exists.


In [22]:
# Cell 23: Load and inspect 2WikiMQA evidence

import json
from collections import Counter

two_wiki_data = load_json(TWO_WIKI_DATA_PATH)

print("num_records =", len(two_wiki_data))
print("type_counts =", Counter(r.get("type", "unknown") for r in two_wiki_data))
print("first_keys =", list(two_wiki_data[0].keys()))

print("\nFirst question:")
print(two_wiki_data[0]["question"])

print("\nFirst answer:")
print(two_wiki_data[0].get("answer", ""))

print("\nFirst evidence preview:")
for i, ev in enumerate(two_wiki_data[0].get("evidence", [])[:5], start=1):
    print(f"{i}.", ev[:500])

num_records = 1000
type_counts = Counter({'bridge_comparison': 250, 'inference': 250, 'comparison': 250, 'compositional': 250})
first_keys = ['type', 'question', 'evidence', 'answer', 'supports']

First question:
Do both films: And Then There Were None (1945 Film) and Langue Sacrée, Langue Parlée have the directors from the same country?

First answer:
yes

First evidence preview:
1. Title: Pauline Auzou. Evidence: Taylor & Francis; January 1997. . p. 199.
2. Title: Martial Law (1991 film). Evidence: The film has yet to arrive onto DVD in the United States.
3. Title: Pauline Auzou. Evidence: Berg; 6 April 1995. . p. 34.
4. Title: Jean Rollin. Evidence: Le temps d'un visage (1990), Jean Rollin. Éd.
5. Title: Até que a Sorte nos Separe. Evidence: Até que a Sorte nos Separe (English: Till Luck Do Us Part) is a 2012 Brazilian comedy film directed by Roberto Santucci and starring Leandro Hassum and Danielle Winits.


In [23]:
# Cell 24: Build one 2WikiMQA input prompt

two_wiki_sample = two_wiki_data[0]

two_wiki_sample_type = two_wiki_sample["type"]
two_wiki_sample_question = two_wiki_sample["question"]
two_wiki_sample_contexts = two_wiki_sample["evidence"]
two_wiki_sample_gt = two_wiki_sample["answer"]

if two_wiki_sample_contexts:
    two_wiki_sample_contexts_text = "\n".join(
        f"{i}: {c}" for i, c in enumerate(two_wiki_sample_contexts, start=1)
    )
    two_wiki_sample_input_prompt = prompt.format(
        question=two_wiki_sample_question,
        context=two_wiki_sample_contexts_text,
    )
else:
    two_wiki_sample_input_prompt = none_prompt.format(
        question=two_wiki_sample_question
    )

print("Sample type:", two_wiki_sample_type)
print("Sample question:", two_wiki_sample_question)
print("Sample gt:", two_wiki_sample_gt)
print("\nPrompt preview:")
print(two_wiki_sample_input_prompt[:3000])

Sample type: bridge_comparison
Sample question: Do both films: And Then There Were None (1945 Film) and Langue Sacrée, Langue Parlée have the directors from the same country?
Sample gt: yes

Prompt preview:

    Given the question and its associated contexts below, please generate a concise, precise answer in English. The answer must strictly adhere to the following guidelines:

    - The answer should be directly relevant to the question.
    - Provide the answer in a clear, straightforward format.
    - Limit your answer to no more than 6 words, focusing on the essential information requested.
    - If the provided contexts do not contain enough information to answer the question, respond with "Information not available" or "Cannot determine from provided context."
    - Do not include any additional tokens, explanations, or information beyond the direct answer.
    - Carefully reason through the question and contexts if the question involes time. 

    QUESTION: Do both films: And T

In [24]:
# Cell 25: Smoke test gpt-oss on one 2WikiMQA sample

two_wiki_sample_response = gpt_oss_chat_completion(
    input_prompt=two_wiki_sample_input_prompt,
    max_new_tokens=256,
)

print("Question:", two_wiki_sample_question)
print("GT:", two_wiki_sample_gt)
print("gpt-oss response:", two_wiki_sample_response)

if is_bad_gpt_oss_response(two_wiki_sample_response):
    print("\nWARNING: Bad response detected. Run debug_gpt_oss_raw_completion if needed.")
else:
    print("\nSmoke test output format looks OK.")

Question: Do both films: And Then There Were None (1945 Film) and Langue Sacrée, Langue Parlée have the directors from the same country?
GT: yes
gpt-oss response: Information not available

Smoke test output format looks OK.


In [25]:
# Cell 26: Define repair-safe resume pipeline for 2WikiMQA

import os
import json
from tqdm import tqdm

def pipeline_resume_repair_2wiki(data, save_path):
    existing_by_question = {}

    if os.path.isfile(save_path):
        with open(save_path, "r", encoding="utf-8") as f:
            existing = json.load(f)

        for r in existing:
            q = r.get("question", "")
            resp = r.get("response", "")
            if q and not is_bad_gpt_oss_response(resp):
                existing_by_question[q] = r

        print("Existing responses:", len(existing))
        print("Valid existing responses kept:", len(existing_by_question))
        print("Bad existing responses to regenerate:", len(existing) - len(existing_by_question))
    else:
        print("No existing 2WikiMQA output found. Starting from scratch.")

    responses = []

    for record in tqdm(data, total=len(data)):
        q_type = record["type"]
        question = record["question"]
        contexts = record["evidence"]
        gt = record["answer"]

        if question in existing_by_question:
            responses.append(existing_by_question[question])
            continue

        if contexts:
            contexts = "\n".join(
                f"{i}: {c}" for i, c in enumerate(contexts, start=1)
            )
            input_prompt = prompt.format(
                question=question,
                context=contexts,
            )
        else:
            input_prompt = none_prompt.format(question=question)

        resp = gpt_oss_chat_completion(
            input_prompt=input_prompt,
            max_new_tokens=256,
        )

        response = {
            "type": q_type,
            "question": question,
            "gt": gt,
            "response": resp,
        }

        responses.append(response)

        with open(save_path, "w", encoding="utf-8") as f:
            json.dump(responses, f, indent=4, ensure_ascii=False)

    return responses

print("2WikiMQA repair-safe resume pipeline ready.")

2WikiMQA repair-safe resume pipeline ready.


In [26]:
# Cell 27: Run answer generation for 2WikiMQA

two_wiki_responses = pipeline_resume_repair_2wiki(
    data=two_wiki_data,
    save_path=TWO_WIKI_SAVE_PATH,
)

print("Finished.")
print("Saved to:", TWO_WIKI_SAVE_PATH)
print("Total responses:", len(two_wiki_responses))

No existing 2WikiMQA output found. Starting from scratch.


100%|██████████| 1000/1000 [50:51<00:00,  3.05s/it]

Finished.
Saved to: /content/drive/MyDrive/final_project/baseline/DATA/KG/answers/wiki_answers_gpt_oss_120b/qwen_agent_responses.json
Total responses: 1000


In [27]:
# Cell 28: Verify saved 2WikiMQA answers

import os
import json
from collections import Counter

assert os.path.isfile(TWO_WIKI_SAVE_PATH), f"Missing output file: {TWO_WIKI_SAVE_PATH}"

two_wiki_saved = load_json(TWO_WIKI_SAVE_PATH)

print("TWO_WIKI_SAVE_PATH =", TWO_WIKI_SAVE_PATH)
print("num_saved =", len(two_wiki_saved))
print("type_counts =", Counter(r.get("type", "unknown") for r in two_wiki_saved))

print("\nFirst saved response:")
print(json.dumps(two_wiki_saved[0], indent=2, ensure_ascii=False)[:3000])

TWO_WIKI_SAVE_PATH = /content/drive/MyDrive/final_project/baseline/DATA/KG/answers/wiki_answers_gpt_oss_120b/qwen_agent_responses.json
num_saved = 1000
type_counts = Counter({'bridge_comparison': 250, 'inference': 250, 'comparison': 250, 'compositional': 250})

First saved response:
{
  "type": "bridge_comparison",
  "question": "Do both films: And Then There Were None (1945 Film) and Langue Sacrée, Langue Parlée have the directors from the same country?",
  "gt": "yes",
  "response": "Information not available"
}


In [28]:
# Cell 29: Check 2WikiMQA output schema consistency

required_keys = {"type", "question", "gt", "response"}

bad_records = []
for i, record in enumerate(two_wiki_saved):
    if set(record.keys()) != required_keys:
        bad_records.append((i, list(record.keys())))

print("Expected keys:", required_keys)
print("Bad records:", len(bad_records))

if bad_records:
    print("First bad record:", bad_records[0])
else:
    print("All 2WikiMQA records have the expected repo-style schema.")

Expected keys: {'type', 'response', 'gt', 'question'}
Bad records: 0
All 2WikiMQA records have the expected repo-style schema.


In [29]:
# Cell 30: Check for analysis-channel outputs in 2WikiMQA

bad_outputs = []
for i, r in enumerate(two_wiki_saved):
    resp = r.get("response", "")
    if is_bad_gpt_oss_response(resp):
        bad_outputs.append((i, r.get("question", ""), resp[:300]))

print("Total saved:", len(two_wiki_saved))
print("Bad outputs:", len(bad_outputs))

if bad_outputs:
    print("\nFirst bad output:")
    print("Index:", bad_outputs[0][0])
    print("Question:", bad_outputs[0][1])
    print("Response preview:", bad_outputs[0][2])
else:
    print("No analysis-channel outputs detected.")

Total saved: 1000
Bad outputs: 0
No analysis-channel outputs detected.


In [30]:
# Cell 33: Final path summary

print("HotpotQA gpt-oss-120b answers:")
print(SAVE_PATH)

print("\n2WikiMQA gpt-oss-120b answers:")
print(TWO_WIKI_SAVE_PATH)

print("\nFiles exist:")
print("HotpotQA:", os.path.isfile(SAVE_PATH))
print("2WikiMQA:", os.path.isfile(TWO_WIKI_SAVE_PATH))

HotpotQA gpt-oss-120b answers:
/content/drive/MyDrive/final_project/baseline/DATA/KG/answers/hotpot_answers_gpt_oss_120b/qwen_agent_responses.json

2WikiMQA gpt-oss-120b answers:
/content/drive/MyDrive/final_project/baseline/DATA/KG/answers/wiki_answers_gpt_oss_120b/qwen_agent_responses.json

Files exist:
HotpotQA: True
2WikiMQA: True
